In [1]:
import pandas as pd
import numpy as np
import os

# Load the dataset
# Adjust the filename if yours doesn't have the "_2" suffix
file_name = "Task 3 and 4_Loan_Data_2.csv"
if not os.path.exists(file_name):
    file_name = "Task 3 and 4_Loan_Data.csv"

df = pd.read_csv(file_name)

# Group the data by FICO score to get 'n' (total count) and 'k' (default count)
grouped = df.groupby('fico_score')['default'].agg(['count', 'sum']).reset_index()
grouped.columns = ['fico_score', 'n', 'k']

# Sort chronologically by FICO score
grouped = grouped.sort_values('fico_score').reset_index(drop=True)

# Preview the grouped data
grouped.head()

,fico_score,n,k
0,408,1,0
1,409,1,1
2,418,1,1
3,425,1,1
4,438,1,1


In [2]:
def calc_log_likelihood(n_bucket, k_bucket):
    """
    Calculates the log-likelihood of a specific bucket.
    """
    if n_bucket == 0:
        return 0.0
    
    p = k_bucket / n_bucket
    ll = 0.0
    
    # Add small safeguards to avoid taking the natural log of 0
    if p > 0:
        ll += k_bucket * np.log(p)
    if p < 1:
        ll += (n_bucket - k_bucket) * np.log(1 - p)
        
    return ll

In [3]:
# Number of unique FICO scores
N = len(grouped)
num_buckets = 5

# Pre-calculate cumulative sums to make math faster
n_cum = np.zeros(N + 1, dtype=int)
k_cum = np.zeros(N + 1, dtype=int)
n_cum[1:] = np.cumsum(grouped['n'].values)
k_cum[1:] = np.cumsum(grouped['k'].values)

# Initialize a DP table with negative infinity
dp = np.full((N + 1, num_buckets + 1), -np.inf)
dp[0, 0] = 0.0  # Base case

# Keep track of where we make our splits
pointers = np.zeros((N + 1, num_buckets + 1), dtype=int)

# Dynamic Programming loop
for i in range(1, N + 1):
    for b in range(1, num_buckets + 1):
        for j in range(i):
            if dp[j, b-1] != -np.inf:
                # Calculate totals for the proposed new bucket
                n_bucket = n_cum[i] - n_cum[j]
                k_bucket = k_cum[i] - k_cum[j]
                
                # Calculate log likelihood for this arrangement
                ll = calc_log_likelihood(n_bucket, k_bucket)
                total_ll = dp[j, b-1] + ll
                
                # If it's the best arrangement we've seen, save it
                if total_ll > dp[i, b]:
                    dp[i, b] = total_ll
                    pointers[i, b] = j

# Backtrack to find the optimal FICO score boundaries
boundaries = []
curr_idx = N
for b in range(num_buckets, 0, -1):
    curr_idx = pointers[curr_idx, b]
    if curr_idx > 0:
        # Save the FICO score where the optimal split occurred
        boundaries.append(grouped['fico_score'].iloc[curr_idx])

boundaries = sorted(boundaries)
print(f"Optimal Log-Likelihood: {dp[N, num_buckets]:.2f}")
print(f"Optimal FICO Boundaries: {boundaries}")

Optimal Log-Likelihood: -4255.38
Optimal FICO Boundaries: [np.int64(521), np.int64(581), np.int64(641), np.int64(697)]


In [4]:
def get_rating(fico_score):
    """
    Maps a FICO score to a risk rating.
    Lower rating (1) = Better credit (Higher FICO).
    Higher rating (5) = Worse credit (Lower FICO).
    """
    if fico_score < boundaries[0]:
        return 5  # Worst credit
    elif fico_score < boundaries[1]:
        return 4
    elif fico_score < boundaries[2]:
        return 3
    elif fico_score < boundaries[3]:
        return 2
    else:
        return 1  # Best credit

# Apply the mapping to our original dataframe
df['rating'] = df['fico_score'].apply(get_rating)

# Verify the buckets by showing the default rates per rating
rating_summary = df.groupby('rating').agg(
    total_borrowers=('default', 'count'),
    total_defaults=('default', 'sum')
)
rating_summary['default_rate'] = (rating_summary['total_defaults'] / rating_summary['total_borrowers'] * 100).round(2).astype(str) + '%'

print("\n--- Final Rating Map Summary ---")
print(rating_summary)


--- Final Rating Map Summary ---
        total_borrowers  total_defaults default_rate
rating                                              
1                  1657              77        4.65%
2                  3197             336       10.51%
3                  3438             703       20.45%
4                  1407             536        38.1%
5                   301             199       66.11%
